## Функция потерь с L1 и L2 регуляризацией


Подумайте, как изменится алгоритм поиска решения для линейной модели, если добавить регуляризацию L1 и L2 к функции потерь. Предоставьте ответ в своём проекте:

**Функция потерь:**

$$
L(w) = \frac{1}{n} \|y - (Xw + b)\|^2 + \lambda_1 \|w\|_1 + \frac{\lambda_2}{2} \|w\|_2^2
$$

**Обновление весов при градиентном спуске:**

$$
w \leftarrow w - \eta \left( \frac{2}{n} X^T (Xw + b - y) + \lambda_2 w + \lambda_1 \cdot \text{sign}(w) \right)
$$

$$
b \leftarrow b - \eta \left( \frac{2}{n}(Xw + b - y) \right)
$$

где:
- $L(w)$ - функция потерь
- $w$ - вектор весов
- $X$ - матрица признаков
- $y$ - вектор целевых значений
- $n$ - количество признаковов
- $\lambda_1$ - коэффициент L1-регуляризации
- $\lambda_2$ - коэффициент L2-регуляризации
- $\eta$ - скорость обучения
- $\text{sign}(w)$ - поэлементная функция знака

## 1) Ответь на вопросы

### Получите аналитическое решение задачи регрессии, используя векторную форму уравнения.

**Функция потерь MSE**:  
$$\mathcal{L}(w) = \frac{1}{n} \|y - Xw\|_2^2$$

**Градиент функции**:  
$$\nabla_w \mathcal{L}(w) = -\frac{2}{n} X^T (y - Xw)$$

**Условие минимума**:  
$$X^T (y - Xw) = 0$$

**Аналитическое решение**:  
$$w = (X^T X)^{-1} X^T y$$

**Особенности**:  
- Решение существует, если $X^T X$ обратима  
- При вырожденной матрице используют псевдообратную матрицу  
- Смещение (intercept) включается как дополнительный признак со значением 1

### Что меняется в решении при добавлении регуляризаций L1 и L2 к функции потерь?


**1. L2-регуляризация (Ridge Regression)**
**Функция потерь**:  
$$\mathcal{L}_{L2}(w) = \frac{1}{n} \|y - Xw\|_2^2 + \lambda \|w\|_2^2$$

**Решение**:  
$$w = (X^T X + \lambda I)^{-1} X^T y$$

**Эффекты**:  
- Стабилизация обращения матрицы  
- Уменьшение величины весов  
- Снижение влияния мультиколлинеарности  

**2. L1-регуляризация (Lasso Regression)**
**Функция потерь**:  
$$\mathcal{L}_{L1}(w) = \frac{1}{n} \|y - Xw\|_2^2 + \lambda \|w\|_1$$

**Решение**:  
Не имеет аналитической формы, решается численными методами



**Компактная запись через soft-thresholding** \
**Решение для каждого веса**:
$$w_j = S_{\frac{n\lambda}{2}} \left( w_j^{OLS} - \sum_{k \neq j} X_j^T X_k w_k \right)$$


*Функция потерь OLS*:
$$\mathcal{L}_{OLS}(w) = \frac{1}{n} \|y - Xw\|_2^2$$

*Аналитическое решение*:
$$w^{OLS} = (X^T X)^{-1} X^T y$$

**где soft-thresholding оператор**:
$$S_{\kappa}(a) = \text{sign}(a) \cdot \max(0, |a| - \kappa)$$

**В координатном спуске**:
$$w_j = \frac{1}{\|X_j\|^2} S_{\frac{n\lambda}{2}} \left( X_j^T(y - X_{-j}w_{-j}) \right)$$

**Практическая интерпретация**

**Пороговое правило**:
- Если $|X_j^T(y - Xw)| > \frac{n\lambda}{2}$: вес ненулевой
- Если $|X_j^T(y - Xw)| \leq \frac{n\lambda}{2}$: вес обнуляется

**Это объясняет**:
- Почему L1 создает разреженность
- Как λ контролирует количество ненулевых весов
- Почему коррелированные признаки могут "конкурировать" за включение в модель

### Объясните, почему для отбора признаков часто используется регуляризация L1. Почему после подгонки модели многие веса оказываются равными 0?


**Почему веса обнуляются**\
**Геометрическая интерпретация**:  
- L1-ограничение образует гиперкуб в пространстве весов  
- Оптимум часто достигается в углах гиперкуба, где координаты равны нулю

**Математическое объяснение**:  
- Субградиент условия оптимальности:  
  $$\frac{\partial \mathcal{L}}{\partial w_i} + \lambda \cdot \text{sign}(w_i) = 0$$
- Если влияние признака мало: $\left|\frac{\partial \mathcal{L}}{\partial w_i}\right| < \lambda$, то вес обнуляется

 *Особенно полезно при большом количестве признаков* 


## 2) Введение — выполните всю предобработку из предыдущего урока

In [12]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler, StandardScaler
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from collections import Counter
from tqdm import tqdm

In [13]:
train_df = pd.read_json("./data/train.json")
train_df

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, Fitness Center, Laundry in...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124000,1.0,3,92bbbf38baadfde0576fc496bd41749c,2016-04-05 03:58:33,There is 700 square feet of recently renovated...,W 171 Street,"[Elevator, Dishwasher, Hardwood Floors]",40.8433,6824800,-73.9396,a61e21da3ba18c7a3d54cfdcc247e1f8,[https://photos.renthop.com/2/6824800_0682be16...,2800,620 W 171 Street,low
124002,1.0,2,5565db9b7cba3603834c4aa6f2950960,2016-04-02 02:25:31,"2 bedroom apartment with updated kitchen, rece...",Broadway,"[Common Outdoor Space, Cats Allowed, Dogs Allo...",40.8198,6813268,-73.9578,8f90e5e10e8a2d7cf997f016d89230eb,[https://photos.renthop.com/2/6813268_1e6fcc32...,2395,3333 Broadway,medium
124004,1.0,1,67997a128056ee1ed7d046bbb856e3c7,2016-04-26 05:42:03,No Brokers Fee * Never Lived 1 Bedroom 1 Bathr...,210 Brighton 15th St,"[Dining Room, Elevator, Pre-War, Laundry in Bu...",40.5765,6927093,-73.9554,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/6927093_93a52104...,1850,210 Brighton 15th St,medium
124008,1.0,2,3c0574a740154806c18bdf1fddd3d966,2016-04-19 02:47:33,Wonderful Bright Chelsea 2 Bedroom apartment o...,West 21st Street,"[Pre-War, Laundry in Unit, Dishwasher, No Fee,...",40.7448,6892816,-74.0017,c3cd45f4381ac371507090e9ffabea80,[https://photos.renthop.com/2/6892816_1a8d087a...,4195,350 West 21st Street,medium


In [16]:
test_df = pd.read_json("./data/test.json")
test_df

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address
0,1.0,1,79780be1514f645d7e6be99a3de696c5,2016-06-11 05:29:41,Large with awesome terrace--accessible via bed...,Suffolk Street,"[Elevator, Laundry in Building, Laundry in Uni...",40.7185,7142618,-73.9865,b1b1852c416d78d7765d746cb1b8921f,[https://photos.renthop.com/2/7142618_1c45a2c8...,2950,99 Suffolk Street
1,1.0,2,0,2016-06-24 06:36:34,Prime Soho - between Bleecker and Houston - Ne...,Thompson Street,"[Pre-War, Dogs Allowed, Cats Allowed]",40.7278,7210040,-74.0000,d0b5648017832b2427eeb9956d966a14,[https://photos.renthop.com/2/7210040_d824cc71...,2850,176 Thompson Street
2,1.0,0,0,2016-06-17 01:23:39,Spacious studio in Prime Location. Cleanbuildi...,Sullivan Street,"[Pre-War, Dogs Allowed, Cats Allowed]",40.7260,7174566,-74.0026,e6472c7237327dd3903b3d6f6a94515a,[https://photos.renthop.com/2/7174566_ba3a35c5...,2295,115 Sullivan Street
3,1.0,2,f9c826104b91d868e69bd25746448c0c,2016-06-21 05:06:02,For immediate access call Bryan.<br /><br />Bo...,Jones Street,"[Hardwood Floors, Dogs Allowed, Cats Allowed]",40.7321,7191391,-74.0028,41735645e0f8f13993c42894023f8e58,[https://photos.renthop.com/2/7191391_8c2f2d49...,2900,23 Jones Street
5,1.0,1,81062936e12ee5fa6cd2b965698e17d5,2016-06-16 07:24:27,Beautiful TRUE 1 bedroom in a luxury building ...,Exchange Place,"[Roof Deck, Doorman, Elevator, Fitness Center,...",40.7054,7171695,-74.0095,a742cf7dd3b2627d83417bc3a1b3ec96,[https://photos.renthop.com/2/7171695_089ffee2...,3254,20 Exchange Place
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124003,1.0,1,bd863d28a6b119ac3bc72d5f27b07f24,2016-04-26 16:09:55,BRAND NEW TO MARKET 1BDR \r107TH AND LEXINGTON...,150 EAST 107TH STREET,[],40.7925,6928108,-73.9454,453d46f8113e1f2c730c2ee5a4469c71,[https://photos.renthop.com/2/6928108_231eb983...,1700,158 EAST 107TH STREET
124005,1.0,2,9174b75c0cd978eb0e5aa93afbad754b,2016-04-21 05:06:19,Convertible 2BR apartment features a brand new...,E 33rd St.,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7456,6906674,-73.9797,2983e45f7e0ad87d677dacd13e362785,[https://photos.renthop.com/2/6906674_9fe899a8...,4195,141 E 33rd St.
124006,1.0,0,0,2016-04-20 01:31:52,"Let's get you in to see this $2,400/mo, recent...",Lexington Avenue,"[Dogs Allowed, Cats Allowed]",40.7416,6897967,-73.9829,e6472c7237327dd3903b3d6f6a94515a,[],2400,95 Lexington Avenue
124007,2.0,2,c90c010e5505365676538e64d02aa1e0,2016-04-08 02:26:45,CooperCooper.com :: Web ID #171357; Access 100...,Park Avenue,"[Doorman, Elevator, Cats Allowed, Dogs Allowed]",40.7485,6842183,-73.9800,6e5c10246156ae5bdcd9b487ca99d96a,[https://photos.renthop.com/2/6842183_b1fe51f4...,6895,41 Park Avenue


In [17]:
train_df = train_df[["bathrooms", "bedrooms", "features", "price"]]
test_df = test_df[["bathrooms", "bedrooms", "features", "price"]]

In [32]:
quantile_1 = train_df["price"].quantile(0.01)
quantile_99 = train_df["price"].quantile(0.99)

train_df = train_df[(train_df["price"] > quantile_1) & (train_df["price"] < quantile_99)]
test_df = test_df[(test_df["price"] > quantile_1) & (test_df["price"] < quantile_99)]

In [34]:
train_df.head()

,bathrooms,bedrooms,features,price
4,1.0,1,"[Dining Room, Pre-War, Laundry in Building, Di...",2400
6,1.0,2,"[Doorman, Elevator, Laundry in Building, Dishw...",3800
9,1.0,2,"[Doorman, Elevator, Laundry in Building, Laund...",3495
10,1.5,3,[],3000
15,1.0,0,"[Doorman, Elevator, Fitness Center, Laundry in...",2795


In [38]:
test_df.head()

,bathrooms,bedrooms,features,price
0,1.0,1,"[Elevator, Laundry in Building, Laundry in Uni...",2950
1,1.0,2,"[Pre-War, Dogs Allowed, Cats Allowed]",2850
2,1.0,0,"[Pre-War, Dogs Allowed, Cats Allowed]",2295
3,1.0,2,"[Hardwood Floors, Dogs Allowed, Cats Allowed]",2900
5,1.0,1,"[Roof Deck, Doorman, Elevator, Fitness Center,...",3254


## 3) Введение в анализ данных, часть 2



### Удалите неиспользуемые символы ([,], ', " и пробел) из столбца.

их нет...
там просто list...

In [43]:
type(test_df["features"][0])

list

In [44]:
# f = lambda x: [s.strip() for s in (x[1:-1]).split(",")]
# test_df["features"].map(f)
# train_df["features"].map(f)
# del(f)

### Получить все значения из каждого списка и объединить результаты в один большой список для всего набора данных. Для этого можно использовать DataFrame.iterrows().


In [46]:
all_features_test = [item for sublist in test_df["features"] for item in sublist]
all_features_train = [item for sublist in train_df["features"] for item in sublist]

In [50]:
all_features_test

['Elevator',
 'Laundry in Building',
 'Laundry in Unit',
 'Dishwasher',
 'Hardwood Floors',
 'Outdoor Space',
 'Pre-War',
 'Dogs Allowed',
 'Cats Allowed',
 'Pre-War',
 'Dogs Allowed',
 'Cats Allowed',
 'Hardwood Floors',
 'Dogs Allowed',
 'Cats Allowed',
 'Roof Deck',
 'Doorman',
 'Elevator',
 'Fitness Center',
 'Pre-War',
 'Laundry in Building',
 'High Speed Internet',
 'Wheelchair Access',
 'Dogs Allowed',
 'Cats Allowed',
 'Cats Allowed',
 'Dogs Allowed',
 'No Fee',
 'Doorman',
 'Elevator',
 'Fitness Center',
 'Laundry In Building',
 'Swimming Pool',
 'Roof Deck',
 'Laundry in Building',
 'Dishwasher',
 'Hardwood Floors',
 'Doorman',
 'Elevator',
 'Laundry in Building',
 'Hardwood Floors',
 'No Fee',
 'No Fee',
 'Fitness Center',
 'Cats Allowed',
 'Dogs Allowed',
 'Swimming Pool',
 'Roof Deck',
 'Doorman',
 'Elevator',
 'Fitness Center',
 'Pre-War',
 'Laundry in Building',
 'Laundry in Unit',
 'Dishwasher',
 'Hardwood Floors',
 'No Fee',
 'Outdoor Space',
 'Doorman',
 'Elevator',
 

### Сколько уникальных значений содержит список результатов?
### Давайте познакомимся с новой библиотекой — Collections. С помощью этого пакета вы сможете эффективно получать количественную статистику по вашим данным.

In [53]:
all_features_test = Counter(all_features_test)
all_features_test

Counter({'Elevator': 38229,
         'Cats Allowed': 35053,
         'Hardwood Floors': 34997,
         'Dogs Allowed': 32608,
         'Doorman': 30800,
         'Dishwasher': 30136,
         'No Fee': 26934,
         'Laundry in Building': 24193,
         'Fitness Center': 19764,
         'Pre-War': 13713,
         'Laundry in Unit': 12690,
         'Roof Deck': 9734,
         'Outdoor Space': 7923,
         'Dining Room': 7303,
         'High Speed Internet': 6189,
         'Balcony': 4510,
         'Swimming Pool': 4266,
         'Laundry In Building': 3874,
         'New Construction': 3717,
         'Terrace': 3201,
         'Exclusive': 3173,
         'Loft': 3098,
         'Garden/Patio': 2868,
         'Common Outdoor Space': 1984,
         'Wheelchair Access': 1945,
         'HARDWOOD': 1387,
         'SIMPLEX': 1373,
         'prewar': 1297,
         'Fireplace': 1240,
         'LOWRISE': 1201,
         'Laundry In Unit': 1153,
         'Reduced Fee': 1105,
         'Garage'

In [54]:
len(all_features_test)

2068

In [55]:
all_features_train = Counter(all_features_train)
all_features_train

Counter({'Elevator': 25375,
         'Hardwood Floors': 23146,
         'Cats Allowed': 23135,
         'Dogs Allowed': 21652,
         'Doorman': 20479,
         'Dishwasher': 20081,
         'No Fee': 17793,
         'Laundry in Building': 16082,
         'Fitness Center': 12989,
         'Pre-War': 8971,
         'Laundry in Unit': 8437,
         'Roof Deck': 6417,
         'Outdoor Space': 5132,
         'Dining Room': 4890,
         'High Speed Internet': 4223,
         'Balcony': 2898,
         'Swimming Pool': 2643,
         'Laundry In Building': 2564,
         'New Construction': 2504,
         'Terrace': 2177,
         'Exclusive': 2075,
         'Loft': 2039,
         'Garden/Patio': 1878,
         'Wheelchair Access': 1311,
         'Common Outdoor Space': 1280,
         'HARDWOOD': 880,
         'SIMPLEX': 873,
         'Fireplace': 837,
         'prewar': 820,
         'LOWRISE': 764,
         'Garage': 736,
         'Laundry Room': 712,
         'Reduced Fee': 686,
     

In [56]:
len(all_features_train)

1539

### Подсчитайте самые популярные функции из нашего огромного списка и выберите 20 лучших на данный момент.

In [58]:
top20_features_train = sorted(all_features_train.items(), key=lambda x: x[1], reverse=True)[:20]
top20_features_train

[('Elevator', 25375),
 ('Hardwood Floors', 23146),
 ('Cats Allowed', 23135),
 ('Dogs Allowed', 21652),
 ('Doorman', 20479),
 ('Dishwasher', 20081),
 ('No Fee', 17793),
 ('Laundry in Building', 16082),
 ('Fitness Center', 12989),
 ('Pre-War', 8971),
 ('Laundry in Unit', 8437),
 ('Roof Deck', 6417),
 ('Outdoor Space', 5132),
 ('Dining Room', 4890),
 ('High Speed Internet', 4223),
 ('Balcony', 2898),
 ('Swimming Pool', 2643),
 ('Laundry In Building', 2564),
 ('New Construction', 2504),
 ('Terrace', 2177)]

### Теперь создайте 20 новых признаков на основе 20 верхних значений: 1, если значение находится в столбце «Признак», в противном случае 0.

In [60]:
for feature, _ in top20_features_train:
    train_df[feature] = train_df["features"].apply(lambda x: int(feature in x))
    test_df[feature] = test_df["features"].apply(lambda x: int(feature in x))

/tmp/ipykernel_34276/531982656.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df[feature] = train_df["features"].apply(lambda x: int(feature in x))
/tmp/ipykernel_34276/531982656.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df[feature] = test_df["features"].apply(lambda x: int(feature in x))
/tmp/ipykernel_34276/531982656.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the ca

In [67]:
test_df.head()

,bathrooms,bedrooms,features,price,Elevator,Hardwood Floors,Cats Allowed,Dogs Allowed,Doorman,Dishwasher,...,Laundry in Unit,Roof Deck,Outdoor Space,Dining Room,High Speed Internet,Balcony,Swimming Pool,Laundry In Building,New Construction,Terrace
0,1.0,1,"[Elevator, Laundry in Building, Laundry in Uni...",2950,1,1,0,0,0,1,...,1,0,1,0,0,0,0,0,0,0
1,1.0,2,"[Pre-War, Dogs Allowed, Cats Allowed]",2850,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1.0,0,"[Pre-War, Dogs Allowed, Cats Allowed]",2295,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1.0,2,"[Hardwood Floors, Dogs Allowed, Cats Allowed]",2900,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
5,1.0,1,"[Roof Deck, Doorman, Elevator, Fitness Center,...",3254,1,0,1,1,1,0,...,0,1,0,0,1,0,0,0,0,0


In [69]:
train_df.head()

,bathrooms,bedrooms,features,price,Elevator,Hardwood Floors,Cats Allowed,Dogs Allowed,Doorman,Dishwasher,...,Laundry in Unit,Roof Deck,Outdoor Space,Dining Room,High Speed Internet,Balcony,Swimming Pool,Laundry In Building,New Construction,Terrace
4,1.0,1,"[Dining Room, Pre-War, Laundry in Building, Di...",2400,0,1,1,1,0,1,...,0,0,0,1,0,0,0,0,0,0
6,1.0,2,"[Doorman, Elevator, Laundry in Building, Dishw...",3800,1,1,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
9,1.0,2,"[Doorman, Elevator, Laundry in Building, Laund...",3495,1,1,0,0,1,1,...,1,0,0,0,0,0,0,0,0,0
10,1.5,3,[],3000,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
15,1.0,0,"[Doorman, Elevator, Fitness Center, Laundry in...",2795,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


### Расширьте наш набор признаков, добавив «ванные комнаты» и «спальни», и создайте специальную переменную feature_list с названиями всех признаков. Теперь у нас 22 значения. Все модели должны быть обучены на этих 22 признаках.

In [72]:
train_df = train_df.drop("features",axis=1)
test_df = test_df.drop("features",axis=1)

X_train = train_df.drop('price', axis=1) 
y_train = train_df['price']  

X_test = test_df.drop('price', axis=1) 
y_test = test_df['price'] 

feature_list = X_train.columns

In [74]:
feature_list

Index(['bathrooms', 'bedrooms', 'Elevator', 'Hardwood Floors', 'Cats Allowed',
       'Dogs Allowed', 'Doorman', 'Dishwasher', 'No Fee',
       'Laundry in Building', 'Fitness Center', 'Pre-War', 'Laundry in Unit',
       'Roof Deck', 'Outdoor Space', 'Dining Room', 'High Speed Internet',
       'Balcony', 'Swimming Pool', 'Laundry In Building', 'New Construction',
       'Terrace'],
      dtype='object')

In [76]:
len(feature_list)

22

In [78]:
X_train.head()

,bathrooms,bedrooms,Elevator,Hardwood Floors,Cats Allowed,Dogs Allowed,Doorman,Dishwasher,No Fee,Laundry in Building,...,Laundry in Unit,Roof Deck,Outdoor Space,Dining Room,High Speed Internet,Balcony,Swimming Pool,Laundry In Building,New Construction,Terrace
4,1.0,1,0,1,1,1,0,1,0,1,...,0,0,0,1,0,0,0,0,0,0
6,1.0,2,1,1,0,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
9,1.0,2,1,1,0,0,1,1,0,1,...,1,0,0,0,0,0,0,0,0,0
10,1.5,3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
15,1.0,0,1,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [80]:
y_train.head()

4     2400
6     3800
9     3495
10    3000
15    2795
Name: price, dtype: int64

## 4) Реализация моделей — Линейная регрессия

### Реализуйте класс Python для алгоритма линейной регрессии с двумя базовыми методами — fit и predict. Используйте стохастический градиентный спуск (SGD) для поиска оптимальных весов модели. Для лучшего понимания рекомендуем реализовать отдельные версии алгоритма с аналитическим решением и нестохастическим градиентным спуском.

In [84]:
class LinearRegression_my():
    def __init__(self, method='Batch_Gradient_Descent', learning_rate=0.01, random_state=42, lambda1=0, lambda2=0, batch_size=32):
        self.method = method
        self.learning_rate = learning_rate
        self.weights = None
        self.bias = None
        self.random_state = random_state
        
        self.lambda1 = lambda1  
        self.lambda2 = lambda2
        self.batch_size = batch_size

    def _compute_gradients(self, X, y, y_pred, weights, m):
        dw = (2/m) * X.T @ (y_pred - y)
        db = (2/m) * np.sum(y_pred - y)
        
        if self.lambda2 > 0:
            dw += self.lambda2 * weights
        
        if self.lambda1 > 0:
            dw += self.lambda1 * np.sign(weights)
        
        return dw, db
    
    def fit(self, X, y, epochs=1000):
        np.random.seed(self.random_state)
        X = X.values if hasattr(X, 'values') else X
        y = y.values if hasattr(y, 'values') else y
        m, n = X.shape
        
        if self.method == 'normal_equation':
            X_with_bias = np.c_[np.ones(X.shape[0]), X]
            self.weights = np.linalg.pinv(X_with_bias.T @ X_with_bias) @ X_with_bias.T @ y
            self.bias = self.weights[0]
            self.weights = self.weights[1:]
            
        elif self.method == "Batch_Gradient_Descent":
            self.weights = np.zeros(n)
            self.bias = 0

            for epoch_num in tqdm(range(epochs)):
                y_pred = X @ self.weights + self.bias
                dw, db = self._compute_gradients(X, y, y_pred, self.weights, m)
                self.weights -= self.learning_rate * dw
                self.bias -= self.learning_rate * db
                
        elif self.method == "Stochastic_Gradient_Descent":
            self.weights = np.zeros(n)
            self.bias = 0

            for epoch_num in tqdm(range(epochs)):
                indices = np.random.permutation(m)
                X_shuffled = X[indices]
                y_shuffled = y[indices]   

                for i in range(m):
                    xi = X_shuffled[i]
                    yi = y_shuffled[i]
                    y_pred = xi @ self.weights + self.bias
                    
                    dw_single = 2 * xi * (y_pred - yi)  
                    db_single = 2 * (y_pred - yi)       
                    
                    if self.lambda2 > 0:
                        dw_single += self.lambda2 * self.weights
                    if self.lambda1 > 0:
                        dw_single += self.lambda1 * np.sign(self.weights)
                    
                    self.weights -= self.learning_rate * dw_single
                    self.bias -= self.learning_rate * db_single
                    
        elif self.method == "MiniBatch_Gradient_Descent":
            self.weights = np.zeros(n)
            self.bias = 0

            for epoch_num in tqdm(range(epochs)):
                indices = np.random.permutation(m)
                X_shuffled = X[indices]
                y_shuffled = y[indices]
                
                for i in range(0, m, self.batch_size):
                    X_batch = X_shuffled[i:i+self.batch_size]
                    y_batch = y_shuffled[i:i+self.batch_size]
                    batch_m = X_batch.shape[0]
                    
                    y_pred = X_batch @ self.weights + self.bias
                    dw, db = self._compute_gradients(X_batch, y_batch, y_pred, self.weights, batch_m)
                    
                    self.weights -= self.learning_rate * dw
                    self.bias -= self.learning_rate * db

    def predict(self, X):
        X = X.values if hasattr(X, 'values') else X
        return X @ self.weights + self.bias

### 2) Что такое детерменистическая модель? Сделайте SGD детерменистическим.

Batch_Gradient_Descent - детерменистическая модель (обучающаяся сразу на ряде разных данных)

### 3) Определите коэффициент R в квадрате (R2) и реализуйте функцию для его расчета.


$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$$
$$SS_{\text{tot}} = \sum_{i=1}^{n} (y_i - \bar{y})^2$$
$$SS_{\text{res}} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$
$n$ - количество наблюдений \
$y$ - истинное значение для i-го наблюдения​\
$\bar{y}$- предсказанное значение для i-го наблюдения\
$\hat{y}$ - среднее арифметическое истинных значений

In [90]:
LinearRegression_my.r2_score = lambda self, X, y: 1 - (np.sum((y - self.predict(X))**2) / np.sum((y - np.mean(y))**2))

### Делайте прогнозы с помощью вашего алгоритма и оцените модель с помощью метрик MAE, RMSE и R2.

In [92]:
LinearRegression_my.mae = lambda self, X, y: abs(y - self.predict(X)).mean()
LinearRegression_my.rmse = lambda self, X, y: np.sqrt(((y - self.predict(X))**2).mean())

### Инициализируйте LinearRegression() из sklearn.linear_model, подгоните модель и спрогнозируйте обучающую и тестовую части, как в предыдущем уроке. 
* Сравните показатели качества и убедитесь, что разница невелика (между вашими реализациями и sklearn).
* Сохраните метрики, как и в предыдущем уроке, в таблице со столбцами «модель», «обучение», «тест для таблицы MAE», «таблица RMSE» и «коэффициент R2».

In [95]:
result_MAE = pd.DataFrame(columns=["model", "train", "test", "metric"])
result_RMSE = pd.DataFrame(columns=["model", "train", "test", "metric"])
result_R2 = pd.DataFrame(columns=["model", "train", "test", "metric"])

def push_metrics(name, model, X_train, y_train, X_test, y_test, 
                result_MAE=result_MAE, result_RMSE=result_RMSE, result_R2=result_R2):
    
    if "Sklearn" in name:
        y_train_predict = model.predict(X_train)
        y_test_predict = model.predict(X_test)
        
        mae_train = mean_absolute_error(y_train, y_train_predict)
        mae_test = mean_absolute_error(y_test, y_test_predict)
        
        rmse_train = np.sqrt(mean_squared_error(y_train, y_train_predict))
        rmse_test = np.sqrt(mean_squared_error(y_test, y_test_predict))
        
        r2_train = r2_score(y_train, y_train_predict)
        r2_test = r2_score(y_test, y_test_predict)
        
    else:
        mae_train = model.mae(X_train, y_train)
        mae_test = model.mae(X_test, y_test)
        
        rmse_train = model.rmse(X_train, y_train)
        rmse_test = model.rmse(X_test, y_test)
        
        r2_train = model.r2_score(X_train, y_train)
        r2_test = model.r2_score(X_test, y_test)
    
    next_index_MAE = len(result_MAE)
    next_index_RMSE = len(result_RMSE)
    next_index_R2 = len(result_R2)
    
    result_MAE.loc[next_index_MAE] = [name, mae_train, mae_test, "MAE"]
    result_RMSE.loc[next_index_RMSE] = [name, rmse_train, rmse_test, "RMSE"]
    result_R2.loc[next_index_R2] = [name, r2_train, r2_test, "R2"]
    
    return result_MAE, result_RMSE, result_R2

In [96]:
models = {
    "Sklearn LinearRegression (Analytical)": LinearRegression(),
    "Sklearn SGDRegressor (Deterministic)": SGDRegressor(random_state=42, max_iter=1000, tol=0.01),
    "My LinearRegression (Normal Equation)": LinearRegression_my(method='normal_equation'),
    "My LinearRegression (Batch GD)": LinearRegression_my(method='Batch_Gradient_Descent'),
    "My LinearRegression (Stochastic GD)": LinearRegression_my(method='Stochastic_Gradient_Descent')
}

print("Сравнение различных реализаций линейной регрессии:")
print("=" * 70)

for name, model in models.items():
    print(f"\n=== {name} ===")
    
    model.fit(X_train, y_train)
    
    if "Sklearn" in name:
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)
        
        mae_train = mean_absolute_error(y_train, y_pred_train)
        rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
        r2_train = r2_score(y_train, y_pred_train)
        
        mae_test = mean_absolute_error(y_test, y_pred_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
        r2_test = r2_score(y_test, y_pred_test)
        
    else:
        mae_train = model.mae(X_train, y_train)
        rmse_train = model.rmse(X_train, y_train)
        r2_train = model.r2_score(X_train, y_train)
        
        mae_test = model.mae(X_test, y_test)
        rmse_test = model.rmse(X_test, y_test)
        r2_test = model.r2_score(X_test, y_test)
    
    print(f"Тренировочные данные: MAE={mae_train:.4f}, RMSE={rmse_train:.4f}, R²={r2_train:.4f}")
    print(f"Тестовые данные:     MAE={mae_test:.4f}, RMSE={rmse_test:.4f}, R²={r2_test:.4f}")
    
    result_MAE, result_RMSE, result_R2 = push_metrics(name, model, X_train, y_train, X_test, y_test)

print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ:")
print("="*70)

print("\nМетрика MAE:")
print(result_MAE.round(4))

print("\nМетрика RMSE:")
print(result_RMSE.round(4))

print("\nМетрика R²:")
print(result_R2.round(4))

Сравнение различных реализаций линейной регрессии:

=== Sklearn LinearRegression (Analytical) ===
Тренировочные данные: MAE=708.7371, RMSE=1027.2629, R²=0.5803
Тестовые данные:     MAE=713.0153, RMSE=1213.5977, R²=0.4052

=== Sklearn SGDRegressor (Deterministic) ===
Тренировочные данные: MAE=707.1975, RMSE=1027.4142, R²=0.5801
Тестовые данные:     MAE=711.4176, RMSE=1212.1024, R²=0.4067

=== My LinearRegression (Normal Equation) ===
Тренировочные данные: MAE=708.7371, RMSE=1027.2629, R²=0.5803
Тестовые данные:     MAE=713.0153, RMSE=1213.5977, R²=0.4052

=== My LinearRegression (Batch GD) ===


100%|██████████████████████████████████████| 1000/1000 [00:01<00:00, 607.73it/s]


Тренировочные данные: MAE=708.9878, RMSE=1028.6358, R²=0.5791
Тестовые данные:     MAE=713.1200, RMSE=1204.7987, R²=0.4138

=== My LinearRegression (Stochastic GD) ===


100%|███████████████████████████████████████| 1000/1000 [03:35<00:00,  4.65it/s]

Тренировочные данные: MAE=863.7366, RMSE=1269.1302, R²=0.3594
Тестовые данные:     MAE=865.0665, RMSE=1382.8212, R²=0.2277
СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ:

Метрика MAE:
                                   model     train      test metric
0  Sklearn LinearRegression (Analytical)  708.7371  713.0153    MAE
1   Sklearn SGDRegressor (Deterministic)  707.1975  711.4176    MAE
2  My LinearRegression (Normal Equation)  708.7371  713.0153    MAE
3         My LinearRegression (Batch GD)  708.9878  713.1200    MAE
4    My LinearRegression (Stochastic GD)  863.7366  865.0665    MAE

Метрика RMSE:
                                   model      train       test metric
0  Sklearn LinearRegression (Analytical)  1027.2629  1213.5977   RMSE
1   Sklearn SGDRegressor (Deterministic)  1027.4142  1212.1024   RMSE
2  My LinearRegression (Normal Equation)  1027.2629  1213.5977   RMSE
3         My LinearRegression (Batch GD)  1028.6358  1204.7987   RMSE
4    My LinearRegression (Stochastic GD)  1269.1302  1382.821

## 5. Реализация регуляризованных моделей — Ridge, Lasso, ElasticNet



In [98]:
models = {
    "My Batch GD (L1)": LinearRegression_my(method='Batch_Gradient_Descent', lambda1=0.01, lambda2=0),
    "My Batch GD (L2)": LinearRegression_my(method='Batch_Gradient_Descent', lambda1=0, lambda2=0.01),
    "My Batch GD (L1+L2)": LinearRegression_my(method='Batch_Gradient_Descent', lambda1=0.01, lambda2=0.01),
    
    "My Stochastic GD (L1)": LinearRegression_my(method='Stochastic_Gradient_Descent', lambda1=0.01, lambda2=0),
    "My Stochastic GD (L2)": LinearRegression_my(method='Stochastic_Gradient_Descent', lambda1=0, lambda2=0.01),
    "My Stochastic GD (L1+L2)": LinearRegression_my(method='Stochastic_Gradient_Descent', lambda1=0.01, lambda2=0.01),
    
    "Sklearn Lasso": Lasso(alpha=0.01, random_state=42),
    "Sklearn Ridge": Ridge(alpha=0.01, random_state=42),
    "Sklearn ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42)
}

print("Сравнение различных реализаций линейной регрессии:")
print("=" * 70)

for name, model in models.items():
    print(f"\n=== {name} ===")
    
    model.fit(X_train, y_train)
    
    if "Sklearn" in name:
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)
        
        mae_train = mean_absolute_error(y_train, y_pred_train)
        rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
        r2_train = r2_score(y_train, y_pred_train)
        
        mae_test = mean_absolute_error(y_test, y_pred_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
        r2_test = r2_score(y_test, y_pred_test)
        
    else:
        mae_train = model.mae(X_train, y_train)
        rmse_train = model.rmse(X_train, y_train)
        r2_train = model.r2_score(X_train, y_train)
        
        mae_test = model.mae(X_test, y_test)
        rmse_test = model.rmse(X_test, y_test)
        r2_test = model.r2_score(X_test, y_test)
    
    print(f"Тренировочные данные: MAE={mae_train:.4f}, RMSE={rmse_train:.4f}, R²={r2_train:.4f}")
    print(f"Тестовые данные:     MAE={mae_test:.4f}, RMSE={rmse_test:.4f}, R²={r2_test:.4f}")
    
    result_MAE, result_RMSE, result_R2 = push_metrics(name, model, X_train, y_train, X_test, y_test)

print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ:")
print("="*70)

print("\nМетрика MAE:")
print(result_MAE.round(4))

print("\nМетрика RMSE:")
print(result_RMSE.round(4))

print("\nМетрика R²:")
print(result_R2.round(4))

Сравнение различных реализаций линейной регрессии:

=== My Batch GD (L1) ===


100%|█████████████████████████████████████| 1000/1000 [00:00<00:00, 1077.84it/s]


Тренировочные данные: MAE=708.9874, RMSE=1028.6374, R²=0.5791
Тестовые данные:     MAE=713.1195, RMSE=1204.7978, R²=0.4138

=== My Batch GD (L2) ===


100%|█████████████████████████████████████| 1000/1000 [00:00<00:00, 1131.30it/s]


Тренировочные данные: MAE=708.8934, RMSE=1029.2567, R²=0.5786
Тестовые данные:     MAE=712.9005, RMSE=1198.0668, R²=0.4203

=== My Batch GD (L1+L2) ===


...

Тренировочные данные: MAE=863.7421, RMSE=1269.1439, R²=0.3593
Тестовые данные:     MAE=865.0710, RMSE=1382.8301, R²=0.2277

=== My Stochastic GD (L2) ===


100%|███████████████████████████████████████| 1000/1000 [04:25<00:00,  3.77it/s]


Тренировочные данные: MAE=862.1839, RMSE=1271.9539, R²=0.3565
Тестовые данные:     MAE=863.2878, RMSE=1377.5579, R²=0.2336

=== My Stochastic GD (L1+L2) ===


100%|███████████████████████████████████████| 1000/1000 [10:46<00:00,  1.55it/s]

Тренировочные данные: MAE=862.1910, RMSE=1271.9699, R²=0.3565
Тестовые данные:     MAE=863.2942, RMSE=1377.5693, R²=0.2336

=== Sklearn Lasso ===
Тренировочные данные: MAE=708.7313, RMSE=1027.2629, R²=0.5803
Тестовые данные:     MAE=713.0093, RMSE=1213.5945, R²=0.4052

=== Sklearn Ridge ===
Тренировочные данные: MAE=708.7371, RMSE=1027.2629, R²=0.5803
Тестовые данные:     MAE=713.0153, RMSE=1213.5973, R²=0.4052

=== Sklearn ElasticNet ===
Тренировочные данные: MAE=708.2902, RMSE=1027.5323, R²=0.5800
Тестовые данные:     MAE=712.3643, RMSE=1204.0805, R²=0.4145
СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ:

Метрика MAE:
                                    model     train      test metric
0   Sklearn LinearRegression (Analytical)  708.7371  713.0153    MAE
1    Sklearn SGDRegressor (Deterministic)  707.1975  711.4176    MAE
2   My LinearRegression (Normal Equation)  708.7371  713.0153    MAE
3          My LinearRegression (Batch GD)  708.9878  713.1200    MAE
4     My LinearRegression (Stochastic GD)  863

## Нормализация признаков

### 1. Сначала напишите несколько примеров того, почему и где нормализация признаков обязательна, и наоборот.


1) в первую очередь в методах на основе растояний (KNN)
2) при вычислении градиентов (все рассмотренные методы)
3) численная стабильность

### 2.Рассмотрим первый из классических методов нормализации — MinMaxScaler. Запишите математическую формулу для этого метода.


$$X_{scaled} = (X - X_{min}) / (X_{max} - X_{min})$$

### 3. Реализуйте собственную функцию или класс для нормализации признаков MinMaxScaler.


In [107]:
class MyMinMaxScaler:
    def __init__(self):
        self.min_ = None
        self.max_ = None
        self.data_range_ = None
    
    def fit(self, X):
        self.min_ = np.min(X, axis=0)
        self.max_ = np.max(X, axis=0)
        self.data_range_ = self.max_ - self.min_
        return self
    
    def transform(self, X):
        if self.min_ is None or self.max_ is None:
            raise ValueError("Scaler must be fitted before transformation")
        data_range = self.data_range_.copy()
        data_range[data_range == 0] = 1
        
        return (X - self.min_) / data_range
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)
    

### 5. Сравните нормализацию признаков с вашим собственным методом и с помощью sklearn.


In [109]:
my_scaler = MyMinMaxScaler()
X_my = my_scaler.fit_transform(X_train)

sklearn_scaler = MinMaxScaler()
X_sklearn = sklearn_scaler.fit_transform(X_train)

np.allclose(X_my, X_sklearn, atol=1e-5)

True

### 6. Повторите шаги от b до e для другого метода нормализации StandardScaler.

$$X_{\text{scaled}} = \frac{X - \mu}{\sigma}$$
$\mu_j = \frac{1}{n} \sum_{i=1}^{n} X_{ij}$ - среднее j-го признака

$\sigma_j = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (X_{ij} - \mu_j)^2}$ - стандартное отклонение j-го признака

In [112]:
class MyStandardScaler:
    def __init__(self):
        self.mean_ = None
        self.scale_ = None  
    
    def fit(self, X):
        self.mean_ = np.mean(X, axis=0)
        self.scale_ = np.std(X, axis=0)
        return self
    
    def transform(self, X):
        if self.mean_ is None or self.scale_ is None:
            raise ValueError("Scaler must be fitted before transformation")
        
        scale = self.scale_.copy()
        scale[scale == 0] = 1
        
        return (X - self.mean_) / scale
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)
    

In [113]:
my_scaler = MyStandardScaler()
X_my = my_scaler.fit_transform(X_train)

sklearn_scaler = StandardScaler()
X_sklearn = sklearn_scaler.fit_transform(X_train)

np.allclose(X_my, X_sklearn, atol=1e-5)

True

## 7. Подгонка пользовательских и sklearn-моделей под нормализованные данные


In [115]:
my_minmax_scaler = MyMinMaxScaler()
my_std_scaler = MyStandardScaler()
sklearn_minmax_scaler = MinMaxScaler()
sklearn_std_scaler = StandardScaler()

X_train_my_minmax = my_minmax_scaler.fit_transform(X_train)
X_test_my_minmax = my_minmax_scaler.transform(X_test)
X_train_my_std = my_std_scaler.fit_transform(X_train)
X_test_my_std = my_std_scaler.transform(X_test)

X_train_sklearn_minmax = sklearn_minmax_scaler.fit_transform(X_train)
X_test_sklearn_minmax = sklearn_minmax_scaler.transform(X_test)
X_train_sklearn_std = sklearn_std_scaler.fit_transform(X_train)
X_test_sklearn_std = sklearn_std_scaler.transform(X_test)

models = {
    "My Batch GD (L1)": LinearRegression_my(method='Batch_Gradient_Descent', lambda1=0.01, lambda2=0),
    "My Batch GD (L2)": LinearRegression_my(method='Batch_Gradient_Descent', lambda1=0, lambda2=0.01),
    "My Batch GD (L1+L2)": LinearRegression_my(method='Batch_Gradient_Descent', lambda1=0.01, lambda2=0.01),
    
    "My Stochastic GD (L1)": LinearRegression_my(method='Stochastic_Gradient_Descent', lambda1=0.01, lambda2=0),
    "My Stochastic GD (L2)": LinearRegression_my(method='Stochastic_Gradient_Descent', lambda1=0, lambda2=0.01),
    "My Stochastic GD (L1+L2)": LinearRegression_my(method='Stochastic_Gradient_Descent', lambda1=0.01, lambda2=0.01),
    
    "Sklearn Lasso": Lasso(alpha=0.01, random_state=42),
    "Sklearn Ridge": Ridge(alpha=0.01, random_state=42),
    "Sklearn ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42)
}

print("Сравнение различных реализаций линейной регрессии:")
print("=" * 70)

for scaler in [0, 1]:  
    for name, model in models.items():
        current_name = name 
        if scaler:
            current_name += " norm = std"  
        else:
            current_name += " norm = minmax"
            
        print(f"\n=== {current_name} ===")
        
        if "My" in name:
            # Свои модели на своих нормализаторах
            if scaler:  # std
                X_train_curr = X_train_my_std  
                X_test_curr = X_test_my_std
            else:  # minmax
                X_train_curr = X_train_my_minmax
                X_test_curr = X_test_my_minmax
        else:
            # Sklearn модели на sklearn нормализаторах  
            if scaler:  # std
                X_train_curr = X_train_sklearn_std  
                X_test_curr = X_test_sklearn_std
            else:  # minmax
                X_train_curr = X_train_sklearn_minmax
                X_test_curr = X_test_sklearn_minmax
        
        model.fit(X_train_curr, y_train)
        
        if "Sklearn" in name:
            y_pred_train = model.predict(X_train_curr)
            y_pred_test = model.predict(X_test_curr)
            
            mae_train = mean_absolute_error(y_train, y_pred_train)
            rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
            r2_train = r2_score(y_train, y_pred_train)
            
            mae_test = mean_absolute_error(y_test, y_pred_test)
            rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
            r2_test = r2_score(y_test, y_pred_test)
            
        else:
            mae_train = model.mae(X_train_curr, y_train)
            rmse_train = model.rmse(X_train_curr, y_train)
            r2_train = model.r2_score(X_train_curr, y_train)
            
            mae_test = model.mae(X_test_curr, y_test)
            rmse_test = model.rmse(X_test_curr, y_test)
            r2_test = model.r2_score(X_test_curr, y_test)
        
        print(f"Тренировочные данные: MAE={mae_train:.4f}, RMSE={rmse_train:.4f}, R²={r2_train:.4f}")
        print(f"Тестовые данные:     MAE={mae_test:.4f}, RMSE={rmse_test:.4f}, R²={r2_test:.4f}")
        
        result_MAE, result_RMSE, result_R2 = push_metrics(current_name, model, X_train_curr, y_train, X_test_curr, y_test)

Сравнение различных реализаций линейной регрессии:

=== My Batch GD (L1) norm = minmax ===


100%|█████████████████████████████████████| 1000/1000 [00:00<00:00, 1056.54it/s]


Тренировочные данные: MAE=885.3194, RMSE=1278.4919, R²=0.3499
Тестовые данные:     MAE=884.0764, RMSE=1269.7893, R²=0.3488

=== My Batch GD (L2) norm = minmax ===


100%|█████████████████████████████████████| 1000/1000 [00:00<00:00, 1027.92it/s]


Тренировочные данные: MAE=891.2906, RMSE=1286.1148, R²=0.3421
Тестовые данные:     MAE=889.9334, RMSE=1277.1397, R²=0.3413

=== My Batch GD (L1+L2) norm = minmax ===


...

Тренировочные данные: MAE=785.6379, RMSE=1124.9997, R²=0.4966
Тестовые данные:     MAE=788.4443, RMSE=1243.1291, R²=0.3759

=== My Stochastic GD (L2) norm = std ===


100%|███████████████████████████████████████| 1000/1000 [04:30<00:00,  3.69it/s]


Тренировочные данные: MAE=784.9306, RMSE=1124.8753, R²=0.4967
Тестовые данные:     MAE=787.7481, RMSE=1242.0094, R²=0.3770

=== My Stochastic GD (L1+L2) norm = std ===


100%|███████████████████████████████████████| 1000/1000 [05:40<00:00,  2.94it/s]

Тренировочные данные: MAE=784.9258, RMSE=1124.8717, R²=0.4967
Тестовые данные:     MAE=787.7435, RMSE=1242.0068, R²=0.3770

=== Sklearn Lasso norm = std ===
Тренировочные данные: MAE=708.7350, RMSE=1027.2629, R²=0.5803
Тестовые данные:     MAE=713.0132, RMSE=1213.5976, R²=0.4052

=== Sklearn Ridge norm = std ===
Тренировочные данные: MAE=708.7371, RMSE=1027.2629, R²=0.5803
Тестовые данные:     MAE=713.0153, RMSE=1213.5976, R²=0.4052

=== Sklearn ElasticNet norm = std ===
Тренировочные данные: MAE=708.6224, RMSE=1027.2737, R²=0.5803
Тестовые данные:     MAE=712.8605, RMSE=1212.2041, R²=0.4066


In [116]:
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ:")
print("="*70)

print("\nМетрика MAE:")
print(result_MAE.round(4))

print("\nМетрика RMSE:")
print(result_RMSE.round(4))

print("\nМетрика R²:")
print(result_R2.round(4))

СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ:

Метрика MAE:
                                     model     train      test metric
0    Sklearn LinearRegression (Analytical)  708.7371  713.0153    MAE
1     Sklearn SGDRegressor (Deterministic)  707.1975  711.4176    MAE
2    My LinearRegression (Normal Equation)  708.7371  713.0153    MAE
3           My LinearRegression (Batch GD)  708.9878  713.1200    MAE
4      My LinearRegression (Stochastic GD)  863.7366  865.0665    MAE
5                         My Batch GD (L1)  708.9874  713.1195    MAE
6                         My Batch GD (L2)  708.8934  712.9005    MAE
7                      My Batch GD (L1+L2)  708.8934  712.9004    MAE
8                    My Stochastic GD (L1)  863.7421  865.0710    MAE
9                    My Stochastic GD (L2)  862.1839  863.2878    MAE
10                My Stochastic GD (L1+L2)  862.1910  863.2942    MAE
11                           Sklearn Lasso  708.7313  713.0093    MAE
12                           Sklearn Ridge  708

## 8. Модели переобучения

In [118]:
X_train_copy = X_train.copy()
X_test_copy = X_test.copy()

selected_features = ["bathrooms", "bedrooms"]
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

poly = PolynomialFeatures(degree=10, include_bias=False)

X_train_poly = poly.fit_transform(X_train_selected)
X_test_poly = poly.transform(X_test_selected)

X_train = pd.DataFrame(X_train_poly, 
                      columns=poly.get_feature_names_out(selected_features),
                      index=X_train.index)

X_test = pd.DataFrame(X_test_poly, 
                     columns=poly.get_feature_names_out(selected_features),
                     index=X_test.index)

In [119]:
X_train.head()

,bathrooms,bedrooms,bathrooms^2,bathrooms bedrooms,bedrooms^2,bathrooms^3,bathrooms^2 bedrooms,bathrooms bedrooms^2,bedrooms^3,bathrooms^4,...,bathrooms^9 bedrooms,bathrooms^8 bedrooms^2,bathrooms^7 bedrooms^3,bathrooms^6 bedrooms^4,bathrooms^5 bedrooms^5,bathrooms^4 bedrooms^6,bathrooms^3 bedrooms^7,bathrooms^2 bedrooms^8,bathrooms bedrooms^9,bedrooms^10
4,1.0,1.0,1.00,1.0,1.0,1.000,1.00,1.0,1.0,1.0000,...,1.000000,1.000000,1.000000,1.000000,1.00000,1.0000,1.000,1.00,1.0,1.0
6,1.0,2.0,1.00,2.0,4.0,1.000,2.00,4.0,8.0,1.0000,...,2.000000,4.000000,8.000000,16.000000,32.00000,64.0000,128.000,256.00,512.0,1024.0
9,1.0,2.0,1.00,2.0,4.0,1.000,2.00,4.0,8.0,1.0000,...,2.000000,4.000000,8.000000,16.000000,32.00000,64.0000,128.000,256.00,512.0,1024.0
10,1.5,3.0,2.25,4.5,9.0,3.375,6.75,13.5,27.0,5.0625,...,115.330078,230.660156,461.320312,922.640625,1845.28125,3690.5625,7381.125,14762.25,29524.5,59049.0
15,1.0,0.0,1.00,0.0,0.0,1.000,0.00,0.0,0.0,1.0000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.0000,0.000,0.00,0.0,0.0


In [120]:
X_test.head()

,bathrooms,bedrooms,bathrooms^2,bathrooms bedrooms,bedrooms^2,bathrooms^3,bathrooms^2 bedrooms,bathrooms bedrooms^2,bedrooms^3,bathrooms^4,...,bathrooms^9 bedrooms,bathrooms^8 bedrooms^2,bathrooms^7 bedrooms^3,bathrooms^6 bedrooms^4,bathrooms^5 bedrooms^5,bathrooms^4 bedrooms^6,bathrooms^3 bedrooms^7,bathrooms^2 bedrooms^8,bathrooms bedrooms^9,bedrooms^10
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,1.0,2.0,1.0,2.0,4.0,1.0,2.0,4.0,8.0,1.0,...,2.0,4.0,8.0,16.0,32.0,64.0,128.0,256.0,512.0,1024.0
2,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,2.0,1.0,2.0,4.0,1.0,2.0,4.0,8.0,1.0,...,2.0,4.0,8.0,16.0,32.0,64.0,128.0,256.0,512.0,1024.0
5,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [121]:
result_MAE_2 = pd.DataFrame(columns=["model", "train", "test", "metric"])
result_RMSE_2 = pd.DataFrame(columns=["model", "train", "test", "metric"])
result_R2_2 = pd.DataFrame(columns=["model", "train", "test", "metric"])

In [122]:
models = {
    "Sklearn LinearRegression (Analytical)": LinearRegression(),
    "Sklearn SGDRegressor (Deterministic)": SGDRegressor(random_state=42, max_iter=1000, tol=0.01),
    "My LinearRegression (Normal Equation)": LinearRegression_my(method='normal_equation'),
    "My LinearRegression (Batch GD)": LinearRegression_my(method='Batch_Gradient_Descent'),
    "My LinearRegression (Stochastic GD)": LinearRegression_my(method='Stochastic_Gradient_Descent')
}

print("Сравнение различных реализаций линейной регрессии:")
print("=" * 70)

for name, model in models.items():
    print(f"\n=== {name} ===")
    
    model.fit(X_train, y_train)
    
    if "Sklearn" in name:
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)
        
        mae_train = mean_absolute_error(y_train, y_pred_train)
        rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
        r2_train = r2_score(y_train, y_pred_train)
        
        mae_test = mean_absolute_error(y_test, y_pred_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
        r2_test = r2_score(y_test, y_pred_test)
        
    else:
        mae_train = model.mae(X_train, y_train)
        rmse_train = model.rmse(X_train, y_train)
        r2_train = model.r2_score(X_train, y_train)
        
        mae_test = model.mae(X_test, y_test)
        rmse_test = model.rmse(X_test, y_test)
        r2_test = model.r2_score(X_test, y_test)
    
    print(f"Тренировочные данные: MAE={mae_train:.4f}, RMSE={rmse_train:.4f}, R²={r2_train:.4f}")
    print(f"Тестовые данные:     MAE={mae_test:.4f}, RMSE={rmse_test:.4f}, R²={r2_test:.4f}")
    
    result_MAE_2, result_RMSE_2, result_R2_2 = push_metrics(name, model, X_train, y_train, X_test, y_test, 
                result_MAE=result_MAE_2, result_RMSE=result_RMSE_2, result_R2=result_R2_2)

Сравнение различных реализаций линейной регрессии:

=== Sklearn LinearRegression (Analytical) ===
Тренировочные данные: MAE=753.6761, RMSE=1070.5990, R²=0.5441
Тестовые данные:     MAE=2779448219069931520.0000, RMSE=751885683818116939776.0000, R²=-228312493787480248588617767563821056.0000

=== Sklearn SGDRegressor (Deterministic) ===
Тренировочные данные: MAE=45913497738466859941888.0000, RMSE=1850981936658247892074496.0000, R²=-1362737171929276531867157798486587213873152.0000
Тестовые данные:     MAE=51345165678059063434831454011392.0000, RMSE=13889696444784668619017939090145280.0000, R²=-77913292093887836214768287431339499515090661112787716676780032.0000

=== My LinearRegression (Normal Equation) ===
Тренировочные данные: MAE=1185.4171, RMSE=1512.6230, R²=0.0899
Тестовые данные:     MAE=105744962167220.5000, RMSE=28605718480608872.0000, R²=-330469791721614306250326016.0000

=== My LinearRegression (Batch GD) ===


  1%|▍                                       | 12/1000 [00:00<00:08, 119.81it/s]/tmp/ipykernel_34276/915070416.py:14: RuntimeWarning: overflow encountered in matmul
  dw = (2/m) * X.T @ (y_pred - y)
/tmp/ipykernel_34276/915070416.py:42: RuntimeWarning: overflow encountered in matmul
  y_pred = X @ self.weights + self.bias
/tmp/ipykernel_34276/915070416.py:42: RuntimeWarning: invalid value encountered in matmul
  y_pred = X @ self.weights + self.bias
100%|██████████████████████████████████████| 1000/1000 [00:06<00:00, 147.97it/s]


Тренировочные данные: MAE=nan, RMSE=nan, R²=1.0000
Тестовые данные:     MAE=nan, RMSE=nan, R²=1.0000

=== My LinearRegression (Stochastic GD) ===


  0%|                                                  | 0/1000 [00:00<?, ?it/s]/tmp/ipykernel_34276/915070416.py:61: RuntimeWarning: overflow encountered in multiply
  dw_single = 2 * xi * (y_pred - yi)
/tmp/ipykernel_34276/915070416.py:69: RuntimeWarning: invalid value encountered in subtract
  self.weights -= self.learning_rate * dw_single
100%|███████████████████████████████████████| 1000/1000 [03:44<00:00,  4.45it/s]

Тренировочные данные: MAE=nan, RMSE=nan, R²=1.0000
Тестовые данные:     MAE=nan, RMSE=nan, R²=1.0000


In [123]:
my_minmax_scaler = MyMinMaxScaler()
my_std_scaler = MyStandardScaler()
sklearn_minmax_scaler = MinMaxScaler()
sklearn_std_scaler = StandardScaler()

X_train_my_minmax = my_minmax_scaler.fit_transform(X_train)
X_test_my_minmax = my_minmax_scaler.transform(X_test)
X_train_my_std = my_std_scaler.fit_transform(X_train)
X_test_my_std = my_std_scaler.transform(X_test)

X_train_sklearn_minmax = sklearn_minmax_scaler.fit_transform(X_train)
X_test_sklearn_minmax = sklearn_minmax_scaler.transform(X_test)
X_train_sklearn_std = sklearn_std_scaler.fit_transform(X_train)
X_test_sklearn_std = sklearn_std_scaler.transform(X_test)

models = {
    "My Batch GD (L1)": LinearRegression_my(method='Batch_Gradient_Descent', lambda1=0.01, lambda2=0),
    "My Batch GD (L2)": LinearRegression_my(method='Batch_Gradient_Descent', lambda1=0, lambda2=0.01),
    "My Batch GD (L1+L2)": LinearRegression_my(method='Batch_Gradient_Descent', lambda1=0.01, lambda2=0.01),
    
    "My Stochastic GD (L1)": LinearRegression_my(method='Stochastic_Gradient_Descent', lambda1=0.01, lambda2=0),
    "My Stochastic GD (L2)": LinearRegression_my(method='Stochastic_Gradient_Descent', lambda1=0, lambda2=0.01),
    "My Stochastic GD (L1+L2)": LinearRegression_my(method='Stochastic_Gradient_Descent', lambda1=0.01, lambda2=0.01),
    
    "Sklearn Lasso": Lasso(alpha=0.01, random_state=42),
    "Sklearn Ridge": Ridge(alpha=0.01, random_state=42),
    "Sklearn ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42)
}

print("Сравнение различных реализаций линейной регрессии:")
print("=" * 70)

for scaler in [0, 1]:  
    for name, model in models.items():
        current_name = name 
        if scaler:
            current_name += " norm = std"  
        else:
            current_name += " norm = minmax"
            
        print(f"\n=== {current_name} ===")
        
        if "My" in name:
            # Свои модели на своих нормализаторах
            if scaler:  # std
                X_train_curr = X_train_my_std  
                X_test_curr = X_test_my_std
            else:  # minmax
                X_train_curr = X_train_my_minmax
                X_test_curr = X_test_my_minmax
        else:
            # Sklearn модели на sklearn нормализаторах  
            if scaler:  # std
                X_train_curr = X_train_sklearn_std  
                X_test_curr = X_test_sklearn_std
            else:  # minmax
                X_train_curr = X_train_sklearn_minmax
                X_test_curr = X_test_sklearn_minmax
        
        model.fit(X_train_curr, y_train)
        
        if "Sklearn" in name:
            y_pred_train = model.predict(X_train_curr)
            y_pred_test = model.predict(X_test_curr)
            
            mae_train = mean_absolute_error(y_train, y_pred_train)
            rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
            r2_train = r2_score(y_train, y_pred_train)
            
            mae_test = mean_absolute_error(y_test, y_pred_test)
            rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
            r2_test = r2_score(y_test, y_pred_test)
            
        else:
            mae_train = model.mae(X_train_curr, y_train)
            rmse_train = model.rmse(X_train_curr, y_train)
            r2_train = model.r2_score(X_train_curr, y_train)
            
            mae_test = model.mae(X_test_curr, y_test)
            rmse_test = model.rmse(X_test_curr, y_test)
            r2_test = model.r2_score(X_test_curr, y_test)
        
        print(f"Тренировочные данные: MAE={mae_train:.4f}, RMSE={rmse_train:.4f}, R²={r2_train:.4f}")
        print(f"Тестовые данные:     MAE={mae_test:.4f}, RMSE={rmse_test:.4f}, R²={r2_test:.4f}")
        
        result_MAE_2, result_RMSE_2, result_R2_2 = push_metrics(current_name, model, X_train_curr, y_train, X_test_curr, y_test, 
                result_MAE=result_MAE_2, result_RMSE=result_RMSE_2, result_R2=result_R2_2)


Сравнение различных реализаций линейной регрессии:

=== My Batch GD (L1) norm = minmax ===


100%|██████████████████████████████████████| 1000/1000 [00:06<00:00, 146.75it/s]


Тренировочные данные: MAE=925.5146, RMSE=1322.3348, R²=0.3045
Тестовые данные:     MAE=364625.4899, RMSE=98386164.3051, R²=-3909255942.0265

=== My Batch GD (L2) norm = minmax ===


100%|██████████████████████████████████████| 1000/1000 [00:06<00:00, 146.54it/s]


Тренировочные данные: MAE=932.9129, RMSE=1331.7888, R²=0.2945
Тестовые данные:     MAE=369394.7540, RMSE=99674373.5873, R²=-4012297027.3723

=== My Batch GD (L1+L2) norm = minmax ===


...

  0%|                                          | 1/1000 [00:00<05:46,  2.89it/s]/tmp/ipykernel_34276/915070416.py:61: RuntimeWarning: overflow encountered in multiply
  dw_single = 2 * xi * (y_pred - yi)
/tmp/ipykernel_34276/915070416.py:59: RuntimeWarning: invalid value encountered in matmul
  y_pred = xi @ self.weights + self.bias
100%|███████████████████████████████████████| 1000/1000 [05:52<00:00,  2.84it/s]


Тренировочные данные: MAE=nan, RMSE=nan, R²=1.0000
Тестовые данные:     MAE=nan, RMSE=nan, R²=1.0000

=== Sklearn Lasso norm = std ===


/home/renat/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.706e+10, tolerance: 1.215e+07
  model = cd_fast.enet_coordinate_descent(


Тренировочные данные: MAE=759.5028, RMSE=1080.5278, R²=0.5356
Тестовые данные:     MAE=7498001090.6682, RMSE=2028329654819.2424, R²=-1661510639506571008.0000

=== Sklearn Ridge norm = std ===
Тренировочные данные: MAE=755.0464, RMSE=1073.1343, R²=0.5419
Тестовые данные:     MAE=80775491504.1848, RMSE=21851070952080.0586, R²=-192828623302265929728.0000

=== Sklearn ElasticNet norm = std ===
Тренировочные данные: MAE=763.6262, RMSE=1088.3081, R²=0.5289
Тестовые данные:     MAE=11070788083.1850, RMSE=2994826585611.0791, R²=-3622174807955181056.0000


/home/renat/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.809e+10, tolerance: 1.215e+07
  model = cd_fast.enet_coordinate_descent(


In [124]:

print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ:")
print("="*70)

print("\nМетрика MAE:")
print(result_MAE_2.round(4))

print("\nМетрика RMSE:")
print(result_RMSE_2.round(4))

print("\nМетрика R²:")
print(result_R2.round(4))

СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ:

Метрика MAE:
                                     model         train          test metric
0    Sklearn LinearRegression (Analytical)  7.536761e+02  2.779448e+18    MAE
1     Sklearn SGDRegressor (Deterministic)  4.591350e+22  5.134517e+31    MAE
2    My LinearRegression (Normal Equation)  1.185417e+03  1.057450e+14    MAE
3           My LinearRegression (Batch GD)           NaN           NaN    MAE
4      My LinearRegression (Stochastic GD)           NaN           NaN    MAE
5           My Batch GD (L1) norm = minmax  9.255146e+02  3.646255e+05    MAE
6           My Batch GD (L2) norm = minmax  9.329129e+02  3.693948e+05    MAE
7        My Batch GD (L1+L2) norm = minmax  9.329251e+02  3.375416e+05    MAE
8      My Stochastic GD (L1) norm = minmax  7.622984e+02  3.182603e+07    MAE
9      My Stochastic GD (L2) norm = minmax  8.145476e+02  1.656306e+06    MAE
10  My Stochastic GD (L1+L2) norm = minmax  8.145622e+02  1.121246e+06    MAE
11             Sklear

## 9. Нативные модели
### Рассчитайте средние и медианные показатели из предыдущего урока и добавьте результаты в итоговый фрейм данных.

In [126]:
X_full = pd.concat([X_train, X_test])
y_full = pd.concat([y_train, y_test])

dummy_configs = [
    ("naive_mean", "mean", X_train, y_train, X_test, y_test),
    ("naive_median", "median", X_train, y_train, X_test, y_test),
    ("naive_mean_full", "mean", X_full, y_full, X_test, y_test),
    ("naive_median_full", "median", X_full, y_full, X_test, y_test),
    ("naive_mean_full_train", "mean", X_full, y_full, X_train, y_train),
    ("naive_median_full_train", "median", X_full, y_full, X_train, y_train)
]

for name, strategy, X_tr, y_tr, X_te, y_te in dummy_configs:
    model = DummyRegressor(strategy=strategy)
    model.fit(X_tr, y_tr)
    
    y_pred_train = model.predict(X_tr)
    y_pred_test = model.predict(X_te)
    
    mae_train = mean_absolute_error(y_tr, y_pred_train)
    mae_test = mean_absolute_error(y_te, y_pred_test)
    
    rmse_train = np.sqrt(mean_squared_error(y_tr, y_pred_train))
    rmse_test = np.sqrt(mean_squared_error(y_te, y_pred_test))
    
    r2_train = r2_score(y_tr, y_pred_train)
    r2_test = r2_score(y_te, y_pred_test)
    
    next_index_MAE = len(result_MAE_2)
    next_index_RMSE = len(result_RMSE_2)
    next_index_R2 = len(result_R2_2)
    
    result_MAE_2.loc[next_index_MAE] = [name, mae_train, mae_test, "MAE"]
    result_RMSE_2.loc[next_index_RMSE] = [name, rmse_train, rmse_test, "RMSE"]
    result_R2_2.loc[next_index_R2] = [name, r2_train, r2_test, "R2"]

## 10. Сравнить результаты

In [128]:
print("СВОДНАЯ ТАБЛИЦА 1 РЕЗУЛЬТАТОВ:")
print("="*70)

print("\nМетрика MAE:")
print(result_MAE.round(4))

print("\nМетрика RMSE:")
print(result_RMSE.round(4))

print("\nМетрика R²:")
print(result_R2.round(4))

СВОДНАЯ ТАБЛИЦА 1 РЕЗУЛЬТАТОВ:

Метрика MAE:
                                     model     train      test metric
0    Sklearn LinearRegression (Analytical)  708.7371  713.0153    MAE
1     Sklearn SGDRegressor (Deterministic)  707.1975  711.4176    MAE
2    My LinearRegression (Normal Equation)  708.7371  713.0153    MAE
3           My LinearRegression (Batch GD)  708.9878  713.1200    MAE
4      My LinearRegression (Stochastic GD)  863.7366  865.0665    MAE
5                         My Batch GD (L1)  708.9874  713.1195    MAE
6                         My Batch GD (L2)  708.8934  712.9005    MAE
7                      My Batch GD (L1+L2)  708.8934  712.9004    MAE
8                    My Stochastic GD (L1)  863.7421  865.0710    MAE
9                    My Stochastic GD (L2)  862.1839  863.2878    MAE
10                My Stochastic GD (L1+L2)  862.1910  863.2942    MAE
11                           Sklearn Lasso  708.7313  713.0093    MAE
12                           Sklearn Ridge  7

In [129]:
print("СВОДНАЯ ТАБЛИЦА 2 РЕЗУЛЬТАТОВ:")
print("="*70)

print("\nМетрика MAE:")
print(result_MAE_2.round(4))

print("\nМетрика RMSE:")
print(result_RMSE_2.round(4))

print("\nМетрика R²:")
print(result_R2_2.round(4))

СВОДНАЯ ТАБЛИЦА 2 РЕЗУЛЬТАТОВ:

Метрика MAE:
                                     model         train          test metric
0    Sklearn LinearRegression (Analytical)  7.536761e+02  2.779448e+18    MAE
1     Sklearn SGDRegressor (Deterministic)  4.591350e+22  5.134517e+31    MAE
2    My LinearRegression (Normal Equation)  1.185417e+03  1.057450e+14    MAE
3           My LinearRegression (Batch GD)           NaN           NaN    MAE
4      My LinearRegression (Stochastic GD)           NaN           NaN    MAE
5           My Batch GD (L1) norm = minmax  9.255146e+02  3.646255e+05    MAE
6           My Batch GD (L2) norm = minmax  9.329129e+02  3.693948e+05    MAE
7        My Batch GD (L1+L2) norm = minmax  9.329251e+02  3.375416e+05    MAE
8      My Stochastic GD (L1) norm = minmax  7.622984e+02  3.182603e+07    MAE
9      My Stochastic GD (L2) norm = minmax  8.145476e+02  1.656306e+06    MAE
10  My Stochastic GD (L1+L2) norm = minmax  8.145622e+02  1.121246e+06    MAE
11             Skle

### Какая модель лучшая?


My Batch GD (L1) - Lasso\
   • Test MAE: 713.1195 | Test R²: 0.4138\
   • Стабильность: разница train/test = 0.58% 



## 11. Задача на сложение

In [192]:
# БЛОК 1: Логарифмическое преобразование целевой переменной
print("="*50)
print("1. ЛОГАРИФМИЧЕСКОЕ ПРЕОБРАЗОВАНИЕ")
print("="*50)

# Создаем копии данных для преобразования
X_train_log = X_train.copy()
X_test_log = X_test.copy()
y_train_log = y_train.copy()
y_test_log = y_test.copy()

# Применяем логарифмирование к целевой переменной
y_train_log = np.log1p(y_train_log)

# Нормализуем признаки
my_minmax_scaler_log = MyMinMaxScaler()
X_train_log_scaled = my_minmax_scaler_log.fit_transform(X_train_log)
X_test_log_scaled = my_minmax_scaler_log.transform(X_test_log)

# Обучаем модель с очень маленькой скоростью обучения и сильной регуляризацией
model_lasso_log = LinearRegression_my(method='Batch_Gradient_Descent', 
                                     lambda1=0.01, lambda2=0, 
                                     learning_rate=0.01)
model_lasso_log.fit(X_train_log_scaled, y_train_log, epochs=1000)

# Предсказания на тренировочных данных
y_train_pred_log = model_lasso_log.predict(X_train_log_scaled)
y_train_pred_log_clipped = np.clip(y_train_pred_log, np.log1p(quanil_1), np.log1p(quanil_99))
y_train_pred = np.expm1(y_train_pred_log_clipped)

# Предсказания на тестовых данных
y_test_pred_log = model_lasso_log.predict(X_test_log_scaled)
y_test_pred_log_clipped = np.clip(y_test_pred_log, np.log1p(quanil_1), np.log1p(quanil_99))
y_test_pred = np.expm1(y_test_pred_log_clipped)

# Метрики для тренировочных данных
mae_train = mean_absolute_error(y_train, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
r2_train = r2_score(y_train, y_train_pred)

# Метрики для тестовых данных
mae_test = mean_absolute_error(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2_test = r2_score(y_test, y_test_pred)

print(f"My Lasso (Batch GD) with log transform:")
print(f"Тренировочные данные: MAE={mae_train:.4f}, RMSE={rmse_train:.4f}, R²={r2_train:.4f}")
print(f"Тестовые данные:     MAE={mae_test:.4f}, RMSE={rmse_test:.4f}, R²={r2_test:.4f}")

1. ЛОГАРИФМИЧЕСКОЕ ПРЕОБРАЗОВАНИЕ


100%|██████████████████████████████████████| 1000/1000 [00:05<00:00, 195.51it/s]

My Lasso (Batch GD) with log transform:
Тренировочные данные: MAE=825.4772, RMSE=1239.0650, R²=0.3893
Тестовые данные:     MAE=827.5148, RMSE=1237.1942, R²=0.3818


In [246]:
# БЛОК 2: Удаление выбросов только из обучающей выборки + логарифмирование
print("\n" + "="*50)
print("2. УДАЛЕНИЕ ВЫБРОСОВ ИЗ ОБУЧАЮЩЕЙ ВЫБОРКИ + ЛОГАРИФМИРОВАНИЕ")
print("="*50)

# Создаем копии данных для очистки
X_train_clean = X_train.copy()
y_train_clean = y_train.copy()
X_test_clean = X_test.copy()
y_test_clean = y_test.copy()

def remove_outliers_iqr(X, y):
    Q1 = y.quantile(0.25)
    Q3 = y.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    mask = (y >= lower_bound) & (y <= upper_bound)
    return X[mask], y[mask]

# Удаляем выбросы только из обучающей выборки
X_train_clean, y_train_clean = remove_outliers_iqr(X_train_clean, y_train_clean)

# Применяем логарифмирование к целевой переменной (только к обучающей)
y_train_clean_log = np.log1p(y_train_clean)

# Нормализуем очищенные данные
my_minmax_scaler_clean = MyMinMaxScaler()
X_train_clean_scaled = my_minmax_scaler_clean.fit_transform(X_train_clean)
X_test_clean_scaled = my_minmax_scaler_clean.transform(X_test_clean)

# Обучаем модель на очищенных данных с логарифмированием
model_lasso_clean = LinearRegression_my(method='Batch_Gradient_Descent', 
                                       lambda1=0.00, lambda2=0.0, 
                                       learning_rate=0.01)
model_lasso_clean.fit(X_train_clean_scaled, y_train_clean_log, epochs=5000)

# Предсказания на тренировочных данных
y_train_pred_log = model_lasso_clean.predict(X_train_clean_scaled)
y_train_pred_log_clipped = np.clip(y_train_pred_log, np.log1p(quanil_1), np.log1p(quanil_99))
y_train_pred = np.expm1(y_train_pred_log_clipped)

# Предсказания на тестовых данных
y_test_pred_log = model_lasso_clean.predict(X_test_clean_scaled)
y_test_pred_log_clipped = np.clip(y_test_pred_log, np.log1p(quanil_1), np.log1p(quanil_99))
y_test_pred = np.expm1(y_test_pred_log_clipped)

# Метрики для тренировочных данных
mae_train_clean = mean_absolute_error(y_train_clean, y_train_pred)
rmse_train_clean = np.sqrt(mean_squared_error(y_train_clean, y_train_pred))
r2_train_clean = r2_score(y_train_clean, y_train_pred)

# Метрики для тестовых данных
mae_test_clean = mean_absolute_error(y_test_clean, y_test_pred)
rmse_test_clean = np.sqrt(mean_squared_error(y_test_clean, y_test_pred))
r2_test_clean = r2_score(y_test_clean, y_test_pred)

print(f"My Lasso (Batch GD) with outliers removal and log transform:")
print(f"Тренировочные данные: MAE={mae_train_clean:.4f}, RMSE={rmse_train_clean:.4f}, R²={r2_train_clean:.4f}")
print(f"Тестовые данные:     MAE={mae_test_clean:.4f}, RMSE={rmse_test_clean:.4f}, R²={r2_test_clean:.4f}")
print(f"Удалено {len(X_train) - len(X_train_clean)} выбросов из обучающих данных")


2. УДАЛЕНИЕ ВЫБРОСОВ ИЗ ОБУЧАЮЩЕЙ ВЫБОРКИ + ЛОГАРИФМИРОВАНИЕ


100%|██████████████████████████████████████| 5000/5000 [00:21<00:00, 237.30it/s]

My Lasso (Batch GD) with outliers removal and log transform:
Тренировочные данные: MAE=670.1463, RMSE=875.8360, R²=0.3559
Тестовые данные:     MAE=842.3581, RMSE=1289.7902, R²=0.3282
Удалено 2589 выбросов из обучающих данных


In [252]:
# БЛОК 3: Мини-пакетный градиентный спуск
print("\n" + "="*50)
print("3. МИНИ-ПАКЕТНЫЙ ГРАДИЕНТНЫЙ СПУСК")
print("="*50)

# Создаем копии данных для мини-пакетного обучения
X_train_mb = X_train.copy()
y_train_mb = y_train.copy()
X_test_mb = X_test.copy()
y_test_mb = y_test.copy()

# Нормализуем данные
my_minmax_scaler_mb = MyMinMaxScaler()
X_train_mb_scaled = my_minmax_scaler_mb.fit_transform(X_train_mb)
X_test_mb_scaled = my_minmax_scaler_mb.transform(X_test_mb)

# Обучаем модель с мини-пакетным градиентным спуском
model_lasso_minibatch = LinearRegression_my(method='MiniBatch_Gradient_Descent', 
                                          lambda1=0.1, lambda2=0.01, 
                                          batch_size=128, learning_rate=0.01)
model_lasso_minibatch.fit(X_train_mb_scaled, y_train_mb)

# Предсказания на тренировочных данных
y_train_pred_mb = model_lasso_minibatch.predict(X_train_mb_scaled)

# Предсказания на тестовых данных
y_test_pred_mb = model_lasso_minibatch.predict(X_test_mb_scaled)

# Метрики для тренировочных данных
mae_train_mb = mean_absolute_error(y_train_mb, y_train_pred_mb)
rmse_train_mb = np.sqrt(mean_squared_error(y_train_mb, y_train_pred_mb))
r2_train_mb = r2_score(y_train_mb, y_train_pred_mb)

# Метрики для тестовых данных
mae_test_mb = mean_absolute_error(y_test_mb, y_test_pred_mb)
rmse_test_mb = np.sqrt(mean_squared_error(y_test_mb, y_test_pred_mb))
r2_test_mb = r2_score(y_test_mb, y_test_pred_mb)

print(f"My Lasso (MiniBatch GD L1):")
print(f"Тренировочные данные: MAE={mae_train_mb:.4f}, RMSE={rmse_train_mb:.4f}, R²={r2_train_mb:.4f}")
print(f"Тестовые данные:     MAE={mae_test_mb:.4f}, RMSE={rmse_test_mb:.4f}, R²={r2_test_mb:.4f}")


3. МИНИ-ПАКЕТНЫЙ ГРАДИЕНТНЫЙ СПУСК


100%|███████████████████████████████████████| 1000/1000 [00:27<00:00, 35.72it/s]

My Lasso (MiniBatch GD L1):
Тренировочные данные: MAE=820.6747, RMSE=1191.6924, R²=0.4351
Тестовые данные:     MAE=14602603.0619, RMSE=3950017005.7188, R²=-6301210338925.9346
